In [ ]:
%load_ext autoreload
%autoreload 2

Steps:

- Pull one sample and its activations.  
- layers.2 has lets say 7x7 activation.
- unroll it, 49 activations. Each one is a vector, which is a column now.
- Each channel is then a `49 x max-n-components` grid. We stack them on top of each other
- So for the MNIST network, that is a `16 x 49 x <each-components-max> + 8 x 196 x <each-components-max>`
- Since its an image, we would keep `196` (the maximum value) as the rows.

- Actually, for now, i can put everything as a row (no need to put new channel in a new row, just put a layer in one row). The calculations are not bad for that.

- After that, for each activation, we find the vector
  - this would be one function simple.
  - it would be called for each activation
- Which activations to choose? we do using threshold. Each point has a threshold for attribution. we can simply make a tensor of the same shape as the normal activations adn we good. Simply use (attr > thres) as a mask.

layers.0 is done, now its time for me to finish this.  

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    otsu_threshold,
    zeros_with_1_at,
)
from pt_to_api.utils import *
from pt_to_api.capture import get_model_internals
from pt_to_api.mnist import (
    SimpleMNIST,
    get_contribs_for_inp_vectorized,
    get_mnist_dataloader,
)
from tqdm import tqdm
import gc

In [ ]:
MODEL_PATH = Path("../../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../../pt-to-api/data/first-input-tens.pt")

DRIVE_PATH = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist")

MAIN_OUT_DIR = (DRIVE_PATH / "collect-patches" / "data")
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
device = "mps"

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))
model = model.to(device)

In [ ]:
MODE = "dark"
if MODE == "light":
    plt.style.use("default")
else:
    plt.style.use("dark_background")

# Helpers

In [ ]:
import torch.nn.functional as F
from torch import nn

def get_indices_of_patches_to_extract(contribs, layer_name, channel, pos_threshes, neg_threshes):
    # print("slice", .shape, "thres", pos_threshes.shape, "neg", neg_threshes.shape)
    contrib_slice = contribs[layer_name][:, channel]
    pos_inds = torch.argwhere(contrib_slice >= pos_threshes)
    neg_inds = torch.argwhere(contrib_slice <= neg_threshes)
    return torch.cat([pos_inds, neg_inds])


def patches_of_single_batch_with_indices(input_act_of_batch, layer, indices):
    op_shape = get_output_shape(
        input_act_of_batch.shape, layer.kernel_size, layer.stride, layer.padding, layer.dilation
    )
    op_r, op_c = op_shape[-2], op_shape[-1]
    b = input_act_of_batch.shape[0]

    patches = F.unfold(
        input_act_of_batch, layer.kernel_size, layer.dilation, layer.padding, layer.stride
    ).reshape(b, -1, op_r, op_c)
    
    return torch.stack([
        patches[ind[0], :, ind[1], ind[2]]
        for ind in indices
    ])


def get_output_shape(input_shape, ksize, stride, padding, dilation):
    B, C, H, W = input_shape
    Ho = (H + 2*padding[0] - dilation[0]*(ksize[0]-1) - 1) // stride[0] + 1
    Wo = (W + 2*padding[1] - dilation[1]*(ksize[1]-1) - 1) // stride[1] + 1
    return (B, C, Ho, Wo)

def get_sampled_patches(patches_ds, num_samples=-1):
    patches_ds = Path(patches_ds)
    patches = []
    for p in patches_ds.glob("*.pt"):
        patches.append(torch.load(p, weights_only=False, map_location="cpu").numpy())
    patches = np.concat(patches)
    samples_idxs = torch.randperm(patches.shape[0]).numpy()
    total = len(samples_idxs)
    to_extract = num_samples if num_samples != -1 else total

    samples = patches[samples_idxs[:to_extract]]
    return samples


def save_weight_and_patches(model, layer_name, channel, src_dir, out_dir, n_samples=-1):
    layer = model.get_submodule(f"{layer_name}")
    weight = layer.weight[channel].clone().detach().cpu()
    weight = weight.reshape(-1).numpy()
    to_save_patches = get_sampled_patches(src_dir, n_samples)

    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    torch.save(to_save_patches, out_dir / "samples.pt")
    torch.save(weight, out_dir / f"weight.pt")

def get_contribs_for_batch(batch, targets, device):
    targs = torch.concat([zeros_with_1_at(10, targ) for targ in targets]).to(device)
    contribs, acts, params = get_contribs_for_inp_vectorized(
        batch, model, targs, "layers.5", device
    )
    return contribs, acts, params


def get_stds_and_bad_combs(collected_contribs):
    all_stds = []
    bad_combs = []

    for r in range(collected_contribs.shape[1]):
        for c in range(collected_contribs.shape[2]):
            sm = collected_contribs[:, r, c]
            all_stds.append(sm.std().item())
            if sm.std() < 2e-4 and sm.mean().abs() < 1e-4:
                bad_combs.append((r, c))
    return all_stds, bad_combs


def get_pos_and_neg_threshes(collected_contribs, bad_combs):
    INF_POS_CONTRIB, INF_NEG_CONTRIB = 2, -2

    pos_threshes = np.zeros_like(collected_contribs[0])
    neg_threshes = np.zeros_like(collected_contribs[0])


    pos_threshes.fill(INF_POS_CONTRIB)
    neg_threshes.fill(INF_NEG_CONTRIB)


    for r in range(collected_contribs.shape[1]):
        for c in range(collected_contribs.shape[2]):
            if (r, c) in bad_combs:
                print("not relevant", r, c)
            else:
                vals = collected_contribs[:, r, c].numpy()
                pvals = [v for v in vals if v > 0]
                nvals = [v for v in vals if v < 0]
                pthres, nthres = otsu_threshold(pvals), otsu_threshold(nvals)
                pos_threshes[r, c] = pthres
                neg_threshes[r, c] = nthres
    pos_threshes, neg_threshes = torch.tensor(pos_threshes), torch.tensor(neg_threshes)
    return pos_threshes, neg_threshes

def get_collected_contribs(dl, layer_key, channel, device):
    collected_contribs = []
    for batch, targets in tqdm(dl.train):
        contribs, _, _ = get_contribs_for_batch(batch, targets, device)
        collected_contribs.append(contribs[layer_key][:, channel])
    collected_contribs = torch.concat(collected_contribs)
    print(f"collected contribs for layer={layer_key} channel={channel}")
    return collected_contribs


def single_cycle(model, layer_key, channel, input_act_key, patches_out_dir, device):
    layer = model.get_submodule(layer_key)
    print("############ start contrib collection")
    dl = get_mnist_dataloader(0.2, bs=256)
    collected_contribs = get_collected_contribs(dl, layer_key, channel, device)

    print("########## find combs")
    all_stds, bad_combs = get_stds_and_bad_combs(collected_contribs)

    print("######## find and save thresholds")
    pos_threshes, neg_threshes = get_pos_and_neg_threshes(collected_contribs, bad_combs)
    torch.save(pos_threshes, patches_out_dir / "pos_threshes.pt")
    torch.save(neg_threshes, patches_out_dir / "neg_threshes.pt")

    import gc
    print("gc: released", gc.collect())


    print("########## start collecting patches")
    main_dl = get_mnist_dataloader(1, bs=1024)
    out_dir = patches_out_dir / "patches"
    out_dir.mkdir(exist_ok=True, parents=True)

    for i, (batch, targets) in enumerate(tqdm(main_dl.train)):
        contribs, acts, params = get_contribs_for_batch(batch, targets, device)
        indices = get_indices_of_patches_to_extract(contribs, layer_key, channel, pos_threshes, neg_threshes)
        patches_of_batch = patches_of_single_batch_with_indices(acts[input_act_key], layer, indices)
        torch.save(patches_of_batch, out_dir / f"{i}.pt")

    print("############# save patches")
    save_weight_and_patches(model, layer_key, channel, out_dir, patches_out_dir, subset_size)

In [ ]:
def get_trimmed_contrib(contrib_tensor, pos_mask, neg_mask):
    contrib_tensor.shape, pos_mask.shape

    # indices where it is greater than 0

    pos_high = contrib_tensor.clone()
    pos_high[pos_high <= 0] = 0
    pos_high[pos_high < pos_mask] = 0

    neg_high = contrib_tensor.clone()
    neg_high[neg_high >= 0] = 0
    neg_high[neg_high > neg_mask] = 0

    return pos_high + neg_high

In [ ]:
from pt_to_api.benchmark.scalers import NormaliseStdScaler
import json

def get_loaded_normaliser(file_path):
    normaliser = NormaliseStdScaler()
    state = load_normaliser_state(file_path)
    normaliser.global_std_ = state["global_std_"]
    return normaliser

def load_normaliser_state(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

def get_component_scalers(components):    
    sc = np.abs(components).max(axis=1)
    return sc

In [ ]:
def show_single_example(
    prev_contribs,
    prev_acts,
    contribs,
    pos_mask,
    neg_mask,
    layer_name,
    model,
    basis_vector_finder,
    show_inputs=True,
    show_raw_contribs=True,
):
    if show_inputs:
        S([prev_acts[0][0], prev_contribs[0][0]], (4, 2), 2, viztype="local", mode=MODE)
        plt.show()

    trimmed_contrib = get_trimmed_contrib(contribs, pos_mask, neg_mask)
    if show_raw_contribs:
        S([trimmed_contrib[0].reshape(trimmed_contrib.shape[1], -1)], (20, 5), mode=MODE)
        plt.show()

    layer = model.get_submodule(layer_name)
    return do_for_one_input(
        trimmed_contrib, prev_acts, layer, basis_vector_finder, # channel_by_code_maxes, channel_by_model
    )
    

def show_batch(
    contribs,
    acts,
    indices_to_show,
    pos_mask,
    neg_mask,
    layer_name,
    prev_layer_name,
    model,
    basis_vector_finder,
    show_inputs=True,
    show_raw_contribs=True,
):
    for i in indices_to_show:
        show_single_example(
            contribs[prev_layer_name][i][None, ...],
            acts[prev_layer_name][i][None, ...],
            contribs[layer_name][i][None, ...],
            pos_mask,
            neg_mask,
            layer_name,
            model,
            basis_vector_finder,
            # channel_by_code_maxes,
            # channel_by_model,
            show_inputs,
            show_raw_contribs,
        )


def do_for_one_input(
    trimmed_contrib, input_act, layer, basis_vector_finder, #channel_by_code_maxes, channel_by_model
):
    kernel = layer.weight.clone().detach().cpu().numpy()
    indices = torch.argwhere(trimmed_contrib != 0)
    patches = patches_of_single_batch_with_indices(input_act, layer, indices)

    image_size = trimmed_contrib.shape[-2:]
    images = [
        prepare_graph_image_for_channel(
            image_size,
            channel,
            kernel,
            indices,
            patches,
            basis_vector_finder,
        )
        for channel in range(kernel.shape[0])
    ]
    return images
    # return stack_as_single_image(images)

def stack_as_single_image(images):
    images = [v.T for v in images]
    images = [v.reshape(v.shape[0], -1) for v in images]
    images = np.concat(images)
    return images


def prepare_graph_image_for_channel(
    out_image_shape,
    channel,
    kernel,
    indices,
    patches,
    basis_vector_finder,
):
    # n_comps = channel_by_code_maxes[channel].shape[0]
    n_comps = basis_vector_finder.num_comps(channel)
    res = np.zeros((*out_image_shape, n_comps), dtype=np.float32)
    chan_kernel = kernel[channel].reshape(-1)
    
    chan_mask = indices[:, 1] == channel
    pws = patches[chan_mask].numpy() * chan_kernel
    # print(indices)
    codes  = basis_vector_finder(patches[chan_mask].numpy(), pws, channel)
    idxs = indices[chan_mask]
    for i in range(len(idxs)):
        ix = idxs[i]
        res[ix[2], ix[3], :] = codes[i]
    return res


# each column is the distribution of codes, we want to find the maxima to normalise them
def _get_codes_maxes(c2run, channel):
    maxes = []
    codes = c2run[channel].codes
    for i in range(codes.shape[1]):
        maxes.append(np.max(np.abs(codes[:, i])))
    maxes = np.array(maxes)
    return maxes

def patches_of_single_batch_with_indices(input_act_of_batch, layer, indices):
    op_shape = get_output_shape(
        input_act_of_batch.shape, layer.kernel_size, layer.stride, layer.padding, layer.dilation
    )
    op_r, op_c = op_shape[-2], op_shape[-1]
    b = input_act_of_batch.shape[0]

    patches = F.unfold(
        input_act_of_batch, layer.kernel_size, layer.dilation, layer.padding, layer.stride
    ).reshape(b, -1, op_r, op_c)

    return torch.stack([
        patches[ind[0], :, ind[2], ind[3]]
        for ind in indices
    ])


# Start

In [ ]:
layer_key = "layers.0"
input_act_key = "x"
layer = model.get_submodule(layer_key)
subset_size = 10_000

In [ ]:
layer = model.get_submodule(layer_key)
dl = get_mnist_dataloader(0.2, bs=256)

In [ ]:
batch, targets = next(iter(dl.train))

In [ ]:
contribs, acts, params = get_contribs_for_batch(batch, targets, device)
contribs = {k: v.clone().detach().cpu() for k,v in contribs.items()}
acts = {k: v.clone().detach().cpu() for k,v in acts.items()}

In [ ]:
contribs["layers.0"].shape

In [ ]:
# now i need the thresholds too though. as a tensor itself.
patches_out_dir = MAIN_OUT_DIR / "layers.2" / "0"

In [ ]:
# create pos mask neg mask
pos_mask = torch.zeros(contribs["layers.0"].shape[1:], dtype=torch.float32)
neg_mask = torch.zeros(contribs["layers.0"].shape[1:], dtype=torch.float32)
n_channels = pos_mask.shape[0]
for chan in range(n_channels):
    chan_dir = MAIN_OUT_DIR / "layers.0" / str(chan)
    pos_mask[chan] = torch.load(chan_dir / "pos_threshes.pt", weights_only=False)
    neg_mask[chan] = torch.load(chan_dir / "neg_threshes.pt", weights_only=False)

In [ ]:
from dataclasses import dataclass
from pt_to_api.benchmark.core import LazySingleRun
from pt_to_api.benchmark.jax.core import autoencoder_from_single_run


@dataclass
class MS:
    comps: int
    seed: int

LAYERS_0_NUM_CHANS = 8
layers_0_chan_to_model_spec = {
    0: MS(2,1),
    1: MS(3,1),
    2: MS(2,29),
    3: MS(4,41),
    4: MS(4,15),
    5: MS(7,3),
    6: MS(2,45),
    7: MS(5, 18),
}

layers_2_chan_to_model_spec = {
    0: MS(14,10),
    1: MS(10,25),
    2: MS(11,24),
    3: MS(10,6),
    4: MS(8,35),
    5: MS(9,38),
    6: MS(12,38),
    7: MS(8,25),
    8: MS(11,36),
    9: MS(12,2),
    10: MS(11,46),
    11: MS(10,36),
    12: MS(11, 8),
    13: MS(14,33),
    14: MS(12,47),
    15: MS(14,27),
}

layer_data_dir = MAIN_OUT_DIR / "layers.0"
channel_by_run = {
    chan: LazySingleRun(layer_data_dir / str(chan) / "runs" / str(model_spec.comps) / f"seed_{model_spec.seed}.pt")
    for chan, model_spec in layers_0_chan_to_model_spec.items()
}
channel_by_model = {chan: autoencoder_from_single_run(run) for chan, run in channel_by_run.items()}
channel_by_code_maxes = {channel: _get_codes_maxes(channel_by_run, channel) for channel in channel_by_model}
channel_by_normaliser = {chan: get_loaded_normaliser(layer_data_dir / str(chan) / "normaliser.json") for chan, model_spec in layers_0_chan_to_model_spec.items()}
channel_by_scalers = {channel: get_component_scalers(run.components) for channel, run in channel_by_run.items()}



In [ ]:
layer_0_channel_by_dl = {
    channel: torch.load(layer_data_dir / str(channel) / "patches-dict-learning.pt", weights_only=False)["dl"] 
    for channel in range(LAYERS_0_NUM_CHANS)
}
layer_0_channel_by_dl_normaliser = {
    channel: get_loaded_normaliser(layer_data_dir / str(chan) / "dict_learn_normaliser_for_patches.json")
    for channel in range(LAYERS_0_NUM_CHANS)
}

For each kernel's output, we render a grid. Now we would need to also normalise the codes somehow (they wont be visible on the same scale otherwise).  
We will use the abs max val for normalization (or should we do abs neg max too? no lets do abs max val only).    


So lets do a grid for one channel now.  
We have the codes samples from run.codes anyways, we can use them. Lets use the 7 kernel thing we have for now (is it needed? not sure but oh well). Technically its too big btw.  


Lets do for 6 only for now? yes

In [ ]:
# S([c.reshape(3,3) for c in channel_by_run[0].components] + [kernel[0][0]], ncols=3, mode=MODE, viztype="local")
# plt.show()

 
# S([c.reshape(3,3) for c in channel_by_run[0].components / kernel[0][0].reshape(-1)] + [kernel[0][0]], ncols=3, mode=MODE, viztype="local")
# plt.show()

In [ ]:
from pt_to_api.utils import mk_rect_on_ax
a = acts['x'][6][0].numpy()
axes = S([a], mode=MODE)
mk_rect_on_ax(axes[0], 5, 13, 3, 3, "white")
mk_rect_on_ax(axes[0], 5, 15, 3, 3, "white")
plt.show()

In [ ]:
# each can use either patches or pws, their choice.  

class AEBasisFinder:
    def __init__(self, channel_by_model, channel_by_normaliser, channel_by_scalers):
        self.channel_by_model = channel_by_model
        self.channel_by_normaliser = channel_by_normaliser
        self.channel_by_scalers = channel_by_scalers

    def num_comps(self, channel):
        return int(self.channel_by_model[channel].decoder.kernel.value.shape[0])

    def __call__(self, patches, pws, channel):
        pws = self.channel_by_normaliser[channel].transform(pws)
        _, codes, _ = self.channel_by_model[channel](pws)
        normalised_codes = codes * self.channel_by_scalers[channel]
        return normalised_codes


class DLBasisFinder:
    def __init__(self, channel_by_dl, channel_by_normaliser):
        self.channel_by_dl = channel_by_dl
        self.channel_by_normaliser = channel_by_normaliser

    def num_comps(self, channel):
        return self.channel_by_dl[channel].components_.shape[0]

    def __call__(self, patches, pws, channel):
        if pws.shape[0] == 0:
            return np.zeros((0, self.channel_by_dl[channel].components_.shape[1]))
        patches = self.channel_by_normaliser[channel].transform(patches)
        codes = self.channel_by_dl[channel].transform(patches)
        # print("codes", codes.shape)
        return codes
    

In [ ]:
dl_finder = DLBasisFinder(layer_0_channel_by_dl, layer_0_channel_by_dl_normaliser)

In [ ]:
dl_finder.channel_by_dl[0].components_.shape

In [ ]:
# now that many comps are painful to see
# but we can simply add them up since they are so disjoint
# we would want a weighted sum kinda ish. 
# mmmmmm. i just need to show the max code idx

In [ ]:
def _get_merged_codes(image):
    result = np.argmax(image, axis=-1)
    result[image.max(axis=-1) == 0.] = -1  # tune threshold
    return result

In [ ]:
res = []

for i in tqdm(range(100)):
    prev_layer_name = "x"
    layer_name = "layers.0"
    images = show_single_example(
        contribs[prev_layer_name][i][None, ...],
        acts[prev_layer_name][i][None, ...],
        contribs[layer_name][i][None, ...],
        pos_mask,
        neg_mask,
        layer_name,
        model,
        dl_finder,
        show_inputs=False,
        show_raw_contribs=False,
    )
    explained_images = [_get_merged_codes(img) for img in images]
    res.append({"act": acts[prev_layer_name][i][0], "basis_images": explained_images})
    # res.append((acts[prev_layer_name][i][0], explained_images))

In [ ]:
from pt_to_api.utils import mk_rect_on_ax
def R(ax, y, x, ky=3, kx=3):
    if MODE == "dark":
        color = "cyan"
    else:
        color = "black"
    mk_rect_on_ax(ax, y, x, ky, kx, color)

def show_examples_of_label(explained_input_acts, channel, label, ksize=(3,3), stride=(2,2), padding=(1,1), max_rows=8):
    basis_images = np.array([r["basis_images"][channel] for r in explained_input_acts])
    input_acts = np.array([r["act"] for r in explained_input_acts])
    idxs = np.argwhere(basis_images == label)
    res = []
    for i in idxs:
        iid, r, c = i
        res.append({"act": input_acts[iid], "r": r, "c": c})

    ncols = 4
    col_sz = 6
    row_sz = 6
    ncols = min(ncols, len(res))
    nrows = min(math.ceil(len(res) / ncols), max_rows)
    figsize = (ncols*col_sz, nrows*row_sz)
    axes = S([i["act"] for i in res[:nrows*ncols]], figsize, ncols, mode=MODE)
    for ax, i in zip(axes, res):
        R(ax, i["r"]*stride[0] - padding[0], i["c"]*stride[1] - padding[1], ksize[0], ksize[1])
    plt.show()
        

Ohk, stuff is generally working. now ive got the mappings of each pixel.  
i know the meaning of every activation after layers.0 yay lols

now, what? I will need a system of grouping/identifying these activation patterns?  
Even when i look at the next layer, im gonna ask, what is the set of input patterns that gave right to this output pixel, along with the identifier for the pattern the kernel allowed.   

so, ill need to assign "groups" / "top level groups" or something, to each basis vector.  
That gives me a reasonable starting point. Although, this is again human way of doing stuff. it wont stay for higher layers.  

The other thing i can do is cdecrase the number of components.  

Hmmm, lastly, merko ek activation hai jese. jispe meko pata hai ke vo ese ek pattern dhundra which is an arc detector. i want to find all the arc detections.  
Actually, considering that we have a lot of data, we can start by enumerating ALL graphs, clustering them (which is really just putting the same graph in one group), and analyse the groups. Now this combinatorially increases in scope thuogh (I already have a lot of states).    

Still, it might be useful to see what "groupings" happen for each graph.   
Its a kind of clustering the outputs based in the journey.  

In [ ]:
show_examples_of_label(res, 2, 19)

In [ ]:
_, axes = plt.subplots(2, 4, figsize=(20,10))
axes = axes.flatten()
for i, img in enumerate(res[0][1]):
    axes[i].imshow(img, cmap=cmap)

plt.show()
# we do have something interesting now
# what are the next steps?

In [ ]:
import seaborn as sns

husl_colors = sns.color_palette("husl", 30)
colors = [(0, 0, 0)] + list(husl_colors)

cmap = mcolors.ListedColormap(colors)
bounds = np.arange(-1.5, 30, 1)
norm = mcolors.BoundaryNorm(bounds, cmap.N)

In [ ]:
_, axes = plt.subplots(2, 4, figsize=(20,10))
axes = axes.flatten()
for i, img in enumerate(images):
    axes[i].imshow(_get_merged_codes(img), cmap=cmap)

plt.show()
# we do have something interesting now
# what are the next steps?

In [ ]:
def get_kernel(model, layer_name, out_chan):
    layer = model.get_submodule(layer_name)
    return layer.weight.data[out_chan].detach().clone().cpu().numpy()

In [ ]:
kernel_l0c0 = get_kernel(model, "layers.0", 0)[0]
S([kernel_l0c0], 2)
S([(c.reshape(3,3)) for c in dl_finder.channel_by_dl[0].components_], 5, 3)

In [ ]:
S([kernel_l0c0 * a[15:18,13:16], a[15:18, 13:16]], viztype="local")

In [ ]:

a = acts['x'][17][0].numpy()
axes = S([a], mode=MODE)
R(axes[0], 15, 13)
# R(axes[0], 7, 21)
# R(axes[0], 1, 11)
# mk_rect_on_ax(axes[0], 15, 13, 3, 3, "white")
# mk_rect_on_ax(axes[0], 7, 21, 3, 3, "white")

plt.show()

In [ ]:
i = 17
prev_layer_name = "x"
layer_name = "layers.0"
images = show_single_example(
    contribs[prev_layer_name][i][None, ...],
    acts[prev_layer_name][i][None, ...],
    contribs[layer_name][i][None, ...],
    pos_mask,
    neg_mask,
    layer_name,
    model,
    AEBasisFinder(channel_by_model, channel_by_normaliser, channel_by_scalers),
    # channel_by_code_maxes,
    # channel_by_model,
    show_inputs=True,
    show_raw_contribs=False,
)
for img in images:
    ncomps = img.shape[-1]
    S([img[:,:,i] for i in range(ncomps)], (20,3), ncomps, mode=MODE)
    plt.show()

# S([images[0][:,:,0], images[0][:,:,1]], mode=MODE)

In [ ]:
S([a[15:18, 13:16], a[7:10,21:24]])

In [ ]:
from pt_to_api.utils import mk_rect_on_ax
a = acts['x'][17][0].numpy()
axes = S([a], mode=MODE)
mk_rect_on_ax(axes[0], 15, 13, 3, 3, "white")
mk_rect_on_ax(axes[0], 7, 21, 3, 3, "white")


plt.show()

In [ ]:
i = 54 
prev_layer_name = "x"
layer_name = "layers.0"
images = show_single_example(
    contribs[prev_layer_name][i][None, ...],
    acts[prev_layer_name][i][None, ...],
    contribs[layer_name][i][None, ...],
    pos_mask,
    neg_mask,
    layer_name,
    model,
    channel_by_scalers,
    channel_by_model,
    show_inputs=True,
    show_raw_contribs=False,
)
for img in images:
    ncomps = img.shape[-1]
    S([img[:,:,i] for i in range(ncomps)], (20,3), ncomps, mode=MODE, viztype="global")
    plt.show()

# S([images[0][:,:,0], images[0][:,:,1]], mode=MODE)

In [ ]:
channel_by_model[0](acts["x"])

In [ ]:
i = 36
prev_layer_name = "x"
layer_name = "layers.0"
images = show_single_example(
    contribs[prev_layer_name][i][None, ...],
    acts[prev_layer_name][i][None, ...],
    contribs[layer_name][i][None, ...],
    pos_mask,
    neg_mask,
    layer_name,
    model,
    channel_by_code_maxes,
    channel_by_model,
    show_inputs=True,
    show_raw_contribs=False,
)
for img in images:
    ncomps = img.shape[-1]
    S([img[:,:,i] for i in range(ncomps)], (20,3), ncomps, mode=MODE)
    plt.show()

# S([images[0][:,:,0], images[0][:,:,1]], mode=MODE)

Ohk we have the view of the first layer. Is it glorious? I dont know ;_;   
Ahahahhahahahahahhahahahhaaaaaaaaa. I now need inputs which are similar lol.   

Hmmmmmmmmmmmmm. Hmmmmmmmmmmmmmmmmm. Hmmmmmmmmmmmmmmmmmmmmm

In [ ]:
n = 50
# indices = torch.argwhere(targets == 1).cpu().numpy()
indices = torch.arange(50)[..., None]
# indices = torch.tensor(list(range(50)))
# print(indices_1)
S([batch[i][0][0].cpu().numpy() for i in indices], (20, 30), 5, ax_titles=[str(i.item()) for i in indices], mode=MODE)
plt.show()

In [ ]:
n = 50
indices = torch.argwhere(targets == 3).cpu().numpy()
# print(indices_1)
S([batch[i][0][0].cpu().numpy() for i in indices], (20, 30), 5, ax_titles=[str(i.item()) for i in indices], mode=MODE)
plt.show()

## Detour

Investigate why kernel 6 is weird (its not thresholding correctly, it has components which are both positive and negative).  

It is evident when we see the patches in this kernel btw.   
The top horizontal line can be positive and negative. this kernel is also good at detecting that.   
The basis vector has a pos/neg value in this case. easy. so mixed signs are possible (i was quite wrong in assuming they were not).  



In [ ]:
from pt_to_api.benchmark import thresholding as TR
def get_thresholds_for_comps(run):
    codes = run.codes
    thresholds = []
    for comp_idx in range(codes.shape[1]):
        thresh_state = TR.threshold_assuming_noise_at_0_with_only_one_side_active(codes[:, comp_idx])
        match thresh_state:
            case TR.Ambiguous() | TR.MixedSign():
                raise TR.ThresholdFailureException(f"failure in finding threshold comp_idx={comp_idx}", thresh_state)
            case TR.NonNegative(threshold = t): 
                thresholds.append(t)
            case TR.NonPositive(threshold = t):
                thresholds.append(t)
    return thresholds

In [ ]:
run6 = layers_0_chan_to_run[6]

In [ ]:
S([c.reshape(3,3) for c in run6.components], 20, 5)

In [ ]:
S([layer.weight.data[6][0].clone().cpu().detach()], 2)

In [ ]:
w = layer.weight.data[6][0].clone().cpu().detach().numpy()
recons = np.array([run6.recon[i].reshape(3,3) for i in range(50)])


# S([run6.recon[i].reshape(3,3) for i in range(50)], (20,12), 10)
recons.shape, w.shape

In [ ]:
S([run6.recon[i].reshape(3,3) for i in range(50)], (20,12), 10)
plt.show()

In [ ]:
ps = recons / w
S([ps[i].reshape(3,3) for i in range(50)], (20,12), 10)

In [ ]:
np.argwhere(run6.codes[:, 0] > 0)
# run6.codes[:, 0][]

In [ ]:
plt.hist(run6.codes[:, 0])

In [ ]:
get_thresholds_for_comps(layers_0_chan_to_run[6])

In [ ]:
indices = torch.argwhere(trimmed_contrib != 0)

# each row is a Kh*Kw patch of the above indices
patches = patches_of_single_batch_with_indices(i0_act_x, layer, indices)

In [ ]:
patches.shape, indices.shape